silver_transform: Silver users_hist (Iceberg + MERGE)

Tarea 2 del DAG. Este notebook es el respaldo manual si Airflow falla.

lee Bronze users , limpia y normaliza, aplica reglas de
calidad (silver.dq_log), deduplica por user_id y hace MERGE en nessie.silver.users_hist
con la columna load_date.

Requiere Bronze de users cargado, y MinIO, Nessie y Spark arriba.

Si nada cambió, no escribe nada. Ejecuta todas las celdas de arriba a abajo.

In [1]:
import os
import sys

# La carpeta jobs/ esta montada en Jupyter (/workspace/jobs) y en Airflow (/opt/airflow/jobs)
for _p in ("/workspace/jobs", "/opt/airflow/jobs"):
    if os.path.isdir(_p):
        sys.path.insert(0, _p)
        break

from pyspark import StorageLevel
from pyspark.errors import AnalysisException
from pyspark.sql import Window
from pyspark.sql import functions as F

import config
import silver_lib

# Parametro: periodos de Bronze a consolidar (por defecto, todos los de config.PERIODS)
PERIODS = sorted(config.PERIODS)
TABLE = silver_lib.table_name(config.SILVER_NS, "users_hist")
print("Periodos:", PERIODS, "| tabla destino:", TABLE)

Periodos: ['2020_H1', '2021_H1'] | tabla destino: nessie.silver.users_hist


In [2]:
spark = silver_lib.get_spark("silver_transform")
silver_lib.ensure_namespaces(spark)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
  user_id BIGINT, display_name STRING, reputation BIGINT, creation_date TIMESTAMP,
  last_access_date TIMESTAMP, views INT, up_votes INT, down_votes INT, location STRING,
  account_id BIGINT, source_period STRING, row_hash STRING, load_date TIMESTAMP
) USING iceberg
""")
print("Tabla lista:", TABLE)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Tabla lista: nessie.silver.users_hist


In [3]:
def read_bronze(spark, periods):
    dfs = []
    for p in periods:
        path = config.bronze_s3a("users", p)
        try:
            dfs.append(spark.read.parquet(path).withColumn("source_period", F.lit(p)))
        except AnalysisException as e:
            raise RuntimeError(f"Falta Bronze de users para {p}: {path}. "
                               "Ejecuta antes la ingesta (bronze_seed.py o bronze_users_dlt.py).") from e
    out = dfs[0]
    for d in dfs[1:]:
        out = out.unionByName(d)
    return out


raw = read_bronze(spark, PERIODS)
raw.printSchema()
print("Filas leidas de Bronze:", raw.count())

root
 |-- Id: long (nullable = true)
 |-- Reputation: binary (nullable = true)
 |-- CreationDate: timestamp (nullable = true)
 |-- DisplayName: binary (nullable = true)
 |-- LastAccessDate: timestamp (nullable = true)
 |-- AboutMe: binary (nullable = true)
 |-- Views: long (nullable = true)
 |-- UpVotes: long (nullable = true)
 |-- DownVotes: long (nullable = true)
 |-- WebsiteUrl: binary (nullable = true)
 |-- Location: binary (nullable = true)
 |-- AccountId: long (nullable = true)
 |-- source_period: string (nullable = false)



Filas leidas de Bronze: 1264551


In [4]:
def _blank_to_null(col):
    return F.when(F.length(F.trim(col)) == 0, F.lit(None)).otherwise(col)


def clean_users(df):
    """Normaliza nombres y tipos. No descarta filas (eso lo decide la capa de calidad)."""
    rep = F.trim(F.col("Reputation").cast("string"))
    return df.select(
        F.col("Id").cast("long").alias("user_id"),
        _blank_to_null(F.col("DisplayName").cast("string")).alias("display_name"),
        # Reputation llega como texto: solo se convierte si es un entero valido
        F.when(rep.rlike("^-?[0-9]+$"), rep.cast("long")).alias("reputation"),
        F.col("CreationDate").alias("creation_date"),
        F.col("LastAccessDate").alias("last_access_date"),
        F.col("Views").cast("int").alias("views"),
        F.col("UpVotes").cast("int").alias("up_votes"),
        F.col("DownVotes").cast("int").alias("down_votes"),
        _blank_to_null(F.col("Location").cast("string")).alias("location"),
        F.col("AccountId").cast("long").alias("account_id"),
        F.col("source_period"),
        rep.alias("reputation_raw"),  # solo para calidad; no llega a la tabla
    )


cleaned = clean_users(raw)
cleaned.drop("reputation_raw").show(5, truncate=False)

+-------+-------------+----------+-----------------------+-----------------------+-----+--------+----------+---------------+----------+-------------+
|user_id|display_name |reputation|creation_date          |last_access_date       |views|up_votes|down_votes|location       |account_id|source_period|
+-------+-------------+----------+-----------------------+-----------------------+-----+--------+----------+---------------+----------+-------------+
|2739132|Enigma       |849       |2013-09-02 08:30:38.467|2024-03-22 06:05:44.493|153  |90      |3         |Mumbai         |3248163   |2020_H1      |
|2739175|Bruno        |143       |2013-09-02 08:43:49.533|2022-01-07 12:12:57.05 |12   |24      |0         |Working at home|3248211   |2020_H1      |
|2739176|vinu         |63        |2013-09-02 08:44:17.907|2021-02-05 11:31:19.847|36   |12      |0         |Kochi, India   |3248212   |2020_H1      |
|2739217|Ankit Verma  |51        |2013-09-02 08:56:45.583|2024-03-24 22:52:50.917|8    |3       |0  

In [5]:
def _valid_expr():
    return (
        F.col("user_id").isNotNull()
        & F.col("creation_date").isNotNull()
        & (F.col("creation_date") <= F.current_timestamp())
        & (F.coalesce(F.col("views"), F.lit(0)) >= 0)
        & (F.coalesce(F.col("up_votes"), F.lit(0)) >= 0)
        & (F.coalesce(F.col("down_votes"), F.lit(0)) >= 0)
    )


def _count_if(cond):
    return F.sum(F.when(cond, 1).otherwise(0))


def quality_stats(df):
    """Una sola pasada sobre los datos. Devuelve la lista de reglas para silver.dq_log."""
    valid = _valid_expr()
    negativos = ((F.col("views") < 0) | (F.col("up_votes") < 0) | (F.col("down_votes") < 0))
    r = df.agg(
        F.count(F.lit(1)).alias("total"),
        _count_if(F.col("user_id").isNull()).alias("user_id_nulo"),
        _count_if(F.col("creation_date").isNull()).alias("creation_date_nula"),
        _count_if(F.col("creation_date") > F.current_timestamp()).alias("fecha_futura"),
        _count_if(negativos).alias("negativos"),
        _count_if(valid).alias("validas"),
        F.countDistinct(F.when(valid, F.col("user_id"))).alias("ids_distintos"),
        _count_if((F.col("reputation_raw") != "") & F.col("reputation").isNull()).alias("rep_no_numerica"),
        _count_if(F.col("location").isNull()).alias("sin_ubicacion"),
    ).first()
    total = r["total"]
    return [
        ("user_id_nulo", r["user_id_nulo"], total, "rechazada"),
        ("creation_date_nula", r["creation_date_nula"], total, "rechazada"),
        ("fecha_futura", r["fecha_futura"], total, "rechazada"),
        ("contadores_negativos", r["negativos"], total, "rechazada"),
        ("duplicados_por_user_id", r["validas"] - r["ids_distintos"], total, "deduplicada"),
        ("reputacion_no_numerica", r["rep_no_numerica"], total, "informativa (queda NULL)"),
        ("ubicacion_vacia", r["sin_ubicacion"], total, "informativa (queda NULL)"),
    ]


checks = quality_stats(cleaned)
for regla, filas, total, accion in checks:
    print(f"{regla:<26} {filas:>10} / {total:<10} {accion}")

user_id_nulo                        0 / 1264551    rechazada
creation_date_nula                  0 / 1264551    rechazada
fecha_futura                        0 / 1264551    rechazada
contadores_negativos                0 / 1264551    rechazada
duplicados_por_user_id         166074 / 1264551    deduplicada
reputacion_no_numerica              0 / 1264551    informativa (queda NULL)
ubicacion_vacia                717470 / 1264551    informativa (queda NULL)


In [6]:
VALUE_COLS = [  # columnas que entran en el hash de cambios
    "display_name", "reputation", "creation_date", "last_access_date",
    "views", "up_votes", "down_votes", "location", "account_id",
]
OUTPUT_COLS = ["user_id"] + VALUE_COLS + ["source_period", "row_hash", "load_date"]


def dedupe(df):
    """Un registro por user_id: el de acceso mas reciente (y, a igualdad, el periodo mas nuevo)."""
    w = Window.partitionBy("user_id").orderBy(
        F.col("last_access_date").desc_nulls_last(), F.col("source_period").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def with_hash_and_load_date(df):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in VALUE_COLS]
    return (df.withColumn("row_hash", F.sha2(F.concat_ws("||", *parts), 256))
              .withColumn("load_date", F.current_timestamp())
              .select(*OUTPUT_COLS))


def changed_rows(df, target):
    """Solo lo NUEVO o MODIFICADO: filas cuyo (user_id, row_hash) no esta en el destino."""
    out = df.join(target.select("user_id", "row_hash"), ["user_id", "row_hash"], "left_anti")
    return out.select(*df.columns)  # el join reordena columnas: se restaura el orden de la tabla


valid = dedupe(cleaned.filter(_valid_expr()))
delta = changed_rows(with_hash_and_load_date(valid), spark.table(TABLE))
delta = delta.persist(StorageLevel.DISK_ONLY)  # se calcula una vez; a disco para no usar memoria
n_delta = delta.count()
print(f"Filas nuevas o modificadas: {n_delta}")

Filas nuevas o modificadas: 0


In [7]:
if n_delta:
    delta.createOrReplaceTempView("users_src")
    spark.sql(f"""
        MERGE INTO {TABLE} t USING users_src s ON t.user_id = s.user_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("MERGE aplicado")
else:
    print("Sin cambios: no se escribe nada en la tabla")
delta.unpersist()
silver_lib.log_dq(spark, TABLE, checks)

Sin cambios: no se escribe nada en la tabla


  DQ nessie.silver.users_hist     user_id_nulo                      0 / 1264551    rechazada
  DQ nessie.silver.users_hist     creation_date_nula                0 / 1264551    rechazada
  DQ nessie.silver.users_hist     fecha_futura                      0 / 1264551    rechazada
  DQ nessie.silver.users_hist     contadores_negativos              0 / 1264551    rechazada
  DQ nessie.silver.users_hist     duplicados_por_user_id       166074 / 1264551    deduplicada
  DQ nessie.silver.users_hist     reputacion_no_numerica            0 / 1264551    informativa (queda NULL)
  DQ nessie.silver.users_hist     ubicacion_vacia              717470 / 1264551    informativa (queda NULL)


In [8]:
t = spark.table(TABLE)
n_rows = t.count()
n_ids = t.select("user_id").distinct().count()
print(f"Filas: {n_rows} | user_id distintos: {n_ids} (deben ser iguales)")
spark.sql(f"SELECT count(*) AS archivos_datos FROM {TABLE}.files").show()
spark.sql(f"SELECT count(*) AS snapshots FROM {TABLE}.snapshots").show()
spark.sql(f"SELECT check_name, failed_rows, total_rows, action FROM {silver_lib.DQ_TABLE} "
          f"WHERE table_name = '{TABLE}'").show(truncate=False)
t.orderBy(F.col("load_date").desc()).select(
    "user_id", "display_name", "reputation", "source_period", "load_date").show(5, truncate=False)

Filas: 1098477 | user_id distintos: 1098477 (deben ser iguales)
+--------------+
|archivos_datos|
+--------------+
|             8|
+--------------+

+---------+
|snapshots|
+---------+
|        1|
+---------+

+----------------------+-----------+----------+------------------------+
|check_name            |failed_rows|total_rows|action                  |
+----------------------+-----------+----------+------------------------+
|contadores_negativos  |0          |1264551   |rechazada               |
|creation_date_nula    |0          |1264551   |rechazada               |
|duplicados_por_user_id|166074     |1264551   |deduplicada             |
|fecha_futura          |0          |1264551   |rechazada               |
|reputacion_no_numerica|0          |1264551   |informativa (queda NULL)|
|ubicacion_vacia       |717470     |1264551   |informativa (queda NULL)|
|user_id_nulo          |0          |1264551   |rechazada               |
+----------------------+-----------+----------+------------

In [9]:
spark.stop()